# Does the happiness fit hold up?

A scratchpad, not a pipeline step. `analysis/fit_happiness.py` is the thing
that writes `results/happiness_fit.json`; this notebook only asks whether the
committed numbers still describe `data/happiness.csv`, and it prints its
answer rather than writing a file.

The kernel starts in the notebook's own directory, so everything below is
resolved from `ROOT` rather than from a bare relative path.

In [ ]:
from pathlib import Path
import csv, json

ROOT = Path.cwd().parent
print(ROOT.name, sorted(p.name for p in ROOT.iterdir() if p.is_dir()))

## The data

25 lines of CSV: one self-reported happiness score per week per condition.

In [ ]:
rows = list(csv.DictReader((ROOT / "data" / "happiness.csv").open()))

series = {}
for row in rows:
    series.setdefault(row["condition"], []).append(
        (float(row["week"]), float(row["happiness"]))
    )

for condition, points in sorted(series.items()):
    print(f"{condition:<12} {len(points):2d} weeks  "
          f"{min(v for _, v in points):.1f}-{max(v for _, v in points):.1f}")

## Refit, and compare against what is committed

`numpy.polyfit` on each condition, rounded the same way the pipeline rounds
it. If a line below says `differs`, `results/happiness_fit.json` is stale and
`analysis/fit_happiness.py` needs rerunning.

In [ ]:
import numpy as np

committed = json.loads((ROOT / "results" / "happiness_fit.json").read_text())["fit"]

for condition, points in sorted(series.items()):
    slope, intercept = np.polyfit([w for w, _ in points], [v for _, v in points], 1)
    slope, intercept = round(float(slope), 4), round(float(intercept), 4)
    was = committed[condition]
    same = (slope, intercept) == (was["slope_per_week"], was["intercept"])
    print(f"{condition:<12} slope {slope:+.4f}/week  intercept {intercept:.4f}  "
          f"{'matches' if same else 'DIFFERS from'} results/")

## Where the plot is

Deliberately not here. The published version of this comparison is
`figures/hello/`, drawn by its own source so that the manuscript, the figure
and the caption stay in step. A notebook is where you decide *what* to plot;
`figures/` is where the plot lives once you have decided.